# Silver Tables to Gold

この Notebook は Lakehouse 上の `silver.City`、`silver.Sale`、`silver.Stock_Item`、`silver.Date` を直接読み込み、Gold スキーマに `FactSales` / `DimCity` / `DimStockItem` / `DimDate` を作成します。

## この Notebook の前提
- 入力テーブル: `silver.City`, `silver.Sale`, `silver.Stock_Item`, `silver.Date`
- 出力テーブル: `gold.FactSales`, `gold.DimCity`, `gold.DimStockItem`, `gold.DimDate`
- `silver.Sale` を Fact の粒度とし、City / Stock Item / Date をディメンション化します
- `silver.Date` は日付の正規化が済んだテーブルであることを前提とします
- `FactSales` は `InvoiceDate` / `DeliveryDate` の `date` 型列を持ち、Power BI では `DimDate[Date]` と直接リレーションする想定です
- `Customer`, `Bill To Customer`, `Salesperson` のディメンションは元データ未提供のため、この Notebook では作成しません

## 実装上の方針
- Gold は Power BI のセマンティックモデルから参照されるため、データ型を明示的に固定します
- 金額・税率・重量は `double` ではなく固定小数点 `decimal` に変換します
- 日付は `date`、キー列は `int`、フラグ列は `boolean`、表示列は `string` に寄せます
- 文字列は不要な前後空白を除去し、Power BI 側の列品質を安定させます

## この Notebook で実施すること
1. Silver テーブルを読み込み、型を補正する
2. Fact が参照するキーだけに絞って `DimCity` と `DimStockItem` を作る
3. `silver.Date` を基に `DimDate` を作る
4. `Sale` を `FactSales` に整形し、`InvoiceDate` / `DeliveryDate` を付与する
5. Gold スキーマに保存し、件数と参照整合性を確認する

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, BooleanType, DecimalType
from pyspark.sql.window import Window

LAKEHOUSE_NAME = "demo_lakehouse"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
DIM_CITY_TABLE = "DimCity"
DIM_STOCK_ITEM_TABLE = "DimStockItem"
DIM_DATE_TABLE = "DimDate"
FACT_SALES_TABLE = "FactSales"

MONEY_DECIMAL_TYPE = DecimalType(19, 4)
RATE_DECIMAL_TYPE = DecimalType(9, 4)
WEIGHT_DECIMAL_TYPE = DecimalType(18, 3)

SOURCE_CITY_TABLE = "City"
SOURCE_SALE_TABLE = "Sale"
SOURCE_STOCK_ITEM_TABLE = "Stock_Item"
SOURCE_DATE_TABLE = "Date"

def build_table_fqn(schema_name, table_name):
    if LAKEHOUSE_NAME:
        return f"{LAKEHOUSE_NAME}.{schema_name}.{table_name}"
    return f"{schema_name}.{table_name}"

CITY_SOURCE_FQN = build_table_fqn(SILVER_SCHEMA, SOURCE_CITY_TABLE)
SALE_SOURCE_FQN = build_table_fqn(SILVER_SCHEMA, SOURCE_SALE_TABLE)
STOCK_ITEM_SOURCE_FQN = build_table_fqn(SILVER_SCHEMA, SOURCE_STOCK_ITEM_TABLE)
DATE_SOURCE_FQN = build_table_fqn(SILVER_SCHEMA, SOURCE_DATE_TABLE)

DIM_CITY_PATH = f"Tables/{GOLD_SCHEMA}/{DIM_CITY_TABLE}"
DIM_STOCK_ITEM_PATH = f"Tables/{GOLD_SCHEMA}/{DIM_STOCK_ITEM_TABLE}"
DIM_DATE_PATH = f"Tables/{GOLD_SCHEMA}/{DIM_DATE_TABLE}"
FACT_SALES_PATH = f"Tables/{GOLD_SCHEMA}/{FACT_SALES_TABLE}"

print("LAKEHOUSE_NAME:", LAKEHOUSE_NAME)
print("SILVER_SCHEMA:", SILVER_SCHEMA)
print("GOLD_SCHEMA:", GOLD_SCHEMA)
print("CITY_SOURCE_FQN:", CITY_SOURCE_FQN)
print("SALE_SOURCE_FQN:", SALE_SOURCE_FQN)
print("STOCK_ITEM_SOURCE_FQN:", STOCK_ITEM_SOURCE_FQN)
print("DATE_SOURCE_FQN:", DATE_SOURCE_FQN)
print("MONEY_DECIMAL_TYPE:", MONEY_DECIMAL_TYPE)
print("RATE_DECIMAL_TYPE:", RATE_DECIMAL_TYPE)
print("WEIGHT_DECIMAL_TYPE:", WEIGHT_DECIMAL_TYPE)

## Silver テーブルの読み込みと型補正
Lakehouse 上の `silver` テーブルを直接読み込み、Power BI 向け Gold モデルで扱いやすいように型を明示的に補正します。
金額系は固定小数点 `decimal`、日付は `date`、キーは `int` に寄せます。

In [ ]:
def read_silver_table(table_fqn):
    return spark.sql(f"SELECT * FROM {table_fqn}")

def normalize_text(column_name):
    return F.when(F.col(column_name).isNull(), F.lit(None)).otherwise(F.trim(F.col(column_name)))

def normalize_product_category(column_name):
    normalized = normalize_text(column_name)
    # Remove a redundant Japanese prefix like "商品カテゴリ:" from category labels.
    return F.when(
        normalized.isNull(),
        F.lit(None)
    ).otherwise(F.trim(F.regexp_replace(normalized, r"^商品カテゴリ[:：]\s*", "")))

def transform_city(df):
    return (
        df
        .withColumn("City_Key", F.col("City_Key").cast(IntegerType()))
        .withColumn("WWI_City_ID", F.col("WWI_City_ID").cast(IntegerType()))
        .withColumn("City", normalize_text("City"))
        .withColumn("State_Province", normalize_text("State_Province"))
        .withColumn("Country", normalize_text("Country"))
        .withColumn("Continent", normalize_text("Continent"))
        .withColumn("Sales_Territory", normalize_text("Sales_Territory"))
        .withColumn("Region", normalize_text("Region"))
        .withColumn("Subregion", normalize_text("Subregion"))
        .withColumn("Latest_Recorded_Population", F.col("Latest_Recorded_Population").cast(IntegerType()))
        .withColumn("Valid_From", F.to_timestamp("Valid_From"))
        .withColumn("Valid_To", F.to_timestamp("Valid_To"))
        .withColumn("Lineage_Key", F.col("Lineage_Key").cast(IntegerType()))
        .withColumn("ProcessedTimestamp", F.to_timestamp("ProcessedTimestamp"))
    )

def transform_sale(df):
    return (
        df
        .withColumn("Sale_Key", F.col("Sale_Key").cast(IntegerType()))
        .withColumn("City_Key", F.col("City_Key").cast(IntegerType()))
        .withColumn("Customer_Key", F.col("Customer_Key").cast(IntegerType()))
        .withColumn("Bill_To_Customer_Key", F.col("Bill_To_Customer_Key").cast(IntegerType()))
        .withColumn("Stock_Item_Key", F.col("Stock_Item_Key").cast(IntegerType()))
        .withColumn("Salesperson_Key", F.col("Salesperson_Key").cast(IntegerType()))
        .withColumn("WWI_Invoice_ID", F.col("WWI_Invoice_ID").cast(IntegerType()))
        .withColumn("Description", normalize_text("Description"))
        .withColumn("Package", normalize_text("Package"))
        .withColumn("Invoice_Date_Key", F.to_date("Invoice_Date_Key"))
        .withColumn("Delivery_Date_Key", F.to_date("Delivery_Date_Key"))
        .withColumn("Quantity", F.col("Quantity").cast(IntegerType()))
        .withColumn("Unit_Price", F.col("Unit_Price").cast(MONEY_DECIMAL_TYPE))
        .withColumn("Tax_Rate", F.col("Tax_Rate").cast(RATE_DECIMAL_TYPE))
        .withColumn("Total_Excluding_Tax", F.col("Total_Excluding_Tax").cast(MONEY_DECIMAL_TYPE))
        .withColumn("Tax_Amount", F.col("Tax_Amount").cast(MONEY_DECIMAL_TYPE))
        .withColumn("Profit", F.col("Profit").cast(MONEY_DECIMAL_TYPE))
        .withColumn("Total_Including_Tax", F.col("Total_Including_Tax").cast(MONEY_DECIMAL_TYPE))
        .withColumn("Total_Dry_Items", F.col("Total_Dry_Items").cast(IntegerType()))
        .withColumn("Total_Chiller_Items", F.col("Total_Chiller_Items").cast(IntegerType()))
        .withColumn("Lineage_Key", F.col("Lineage_Key").cast(IntegerType()))
        .withColumn("ProcessedTimestamp", F.to_timestamp("ProcessedTimestamp"))
    )

def transform_stock_item(df):
    return (
        df
        .withColumn("Stock_Item_Key", F.col("Stock_Item_Key").cast(IntegerType()))
        .withColumn("WWI_Stock_Item_ID", F.col("WWI_Stock_Item_ID").cast(IntegerType()))
        .withColumn("Stock_Item", normalize_text("Stock_Item"))
        .withColumn("Color", normalize_text("Color"))
        .withColumn("Selling_Package", normalize_text("Selling_Package"))
        .withColumn("Buying_Package", normalize_text("Buying_Package"))
        .withColumn("Brand", normalize_text("Brand"))
        .withColumn("Size", normalize_text("Size"))
        .withColumn("Lead_Time_Days", F.col("Lead_Time_Days").cast(IntegerType()))
        .withColumn("Quantity_Per_Outer", F.col("Quantity_Per_Outer").cast(IntegerType()))
        .withColumn("Is_Chiller_Stock", F.col("Is_Chiller_Stock").cast(BooleanType()))
        .withColumn("Barcode", normalize_text("Barcode"))
        .withColumn("Tax_Rate", F.col("Tax_Rate").cast(RATE_DECIMAL_TYPE))
        .withColumn("Unit_Price", F.col("Unit_Price").cast(MONEY_DECIMAL_TYPE))
        .withColumn("Recommended_Retail_Price", F.col("Recommended_Retail_Price").cast(MONEY_DECIMAL_TYPE))
        .withColumn("Typical_Weight_Per_Unit", F.col("Typical_Weight_Per_Unit").cast(WEIGHT_DECIMAL_TYPE))
        .withColumn("Photo", normalize_text("Photo"))
        .withColumn("Valid_From", F.to_timestamp("Valid_From"))
        .withColumn("Valid_To", F.to_timestamp("Valid_To"))
        .withColumn("Lineage_Key", F.col("Lineage_Key").cast(IntegerType()))
        .withColumn("Product_Category", normalize_product_category("Product_Category"))
    )

def transform_date(df):
    return (
        df
        .withColumn("Date", F.to_date("Date"))
        .withColumn("Day_Number", F.col("Day_Number").cast(IntegerType()))
        .withColumn("Day", F.col("Day").cast(IntegerType()))
        .withColumn("Month", normalize_text("Month"))
        .withColumn("Short_Month", normalize_text("Short_Month"))
        .withColumn("Calendar_Month_Number", F.col("Calendar_Month_Number").cast(IntegerType()))
        .withColumn("Calendar_Month_Label", normalize_text("Calendar_Month_Label"))
        .withColumn("Calendar_Year", F.col("Calendar_Year").cast(IntegerType()))
        .withColumn("Calendar_Year_Label", normalize_text("Calendar_Year_Label"))
        .withColumn("Fiscal_Month_Number", F.col("Fiscal_Month_Number").cast(IntegerType()))
        .withColumn("Fiscal_Month_Label", normalize_text("Fiscal_Month_Label"))
        .withColumn("Fiscal_Year", F.col("Fiscal_Year").cast(IntegerType()))
        .withColumn("Fiscal_Year_Label", normalize_text("Fiscal_Year_Label"))
        .withColumn("ISO_Week_Number", F.col("ISO_Week_Number").cast(IntegerType()))
    )

## 入力データの概要確認
Fact の件数、日付範囲、空の Delivery Date 件数を確認します。

In [ ]:
input_profile = [
    ("City", city_silver.count(), city_silver.select('City_Key').distinct().count()),
    ("Sale", sale_silver.count(), sale_silver.select('Sale_Key').distinct().count()),
    ("Stock_Item", stock_item_silver.count(), stock_item_silver.select('Stock_Item_Key').distinct().count()),
    ("Date", date_silver.count(), date_silver.select('Date').distinct().count())
]
display(spark.createDataFrame(input_profile, ['TableName', 'RowCount', 'DistinctPrimaryKeyCount']))

sale_date_summary = sale_silver.agg(
    F.min('Invoice_Date_Key').alias('MinInvoiceDate'),
    F.max('Invoice_Date_Key').alias('MaxInvoiceDate'),
    F.min('Delivery_Date_Key').alias('MinDeliveryDate'),
    F.max('Delivery_Date_Key').alias('MaxDeliveryDate'),
    F.sum(F.when(F.col('Delivery_Date_Key').isNull(), 1).otherwise(0)).alias('NullDeliveryDateCount')
)
display(sale_date_summary)

date_summary = date_silver.agg(
    F.min('Date').alias('MinDate'),
    F.max('Date').alias('MaxDate'),
    F.countDistinct('Date').alias('DistinctDateCount')
)
display(date_summary)

fact_date_preview = (
    sale_silver
    .select(
        F.col('Invoice_Date_Key').alias('InvoiceDate'),
        F.col('Delivery_Date_Key').alias('DeliveryDate')
    )
    .distinct()
    .orderBy('InvoiceDate', 'DeliveryDate')
)
display(fact_date_preview.limit(20))

## Gold ディメンションの作成
`DimCity` と `DimStockItem` は、Fact が参照するキーだけに絞って作成します。

Power BI のセマンティックモデルで扱いやすいように、Gold 出力時にもキー列・数値列・文字列列の型を明示的に固定します。

In [ ]:
referenced_city_keys = sale_silver.select('City_Key').distinct()
referenced_stock_item_keys = sale_silver.select('Stock_Item_Key').distinct()

city_window = Window.partitionBy('City_Key').orderBy(F.col('Valid_To').desc_nulls_last(), F.col('ProcessedTimestamp').desc_nulls_last())
stock_item_window = Window.partitionBy('Stock_Item_Key').orderBy(F.col('Valid_To').desc_nulls_last(), F.col('Valid_From').desc_nulls_last())

dim_city = (
    city_silver
    .join(referenced_city_keys, on='City_Key', how='inner')
    .withColumn('RowPriority', F.row_number().over(city_window))
    .filter(F.col('RowPriority') == 1)
    .select(
        F.col('City_Key').cast(IntegerType()).alias('CityKey'),
        F.col('WWI_City_ID').cast(IntegerType()).alias('WWICityId'),
        normalize_text('City').alias('CityName'),
        normalize_text('State_Province').alias('StateProvince'),
        normalize_text('Country').alias('Country'),
        normalize_text('Continent').alias('Continent'),
        normalize_text('Sales_Territory').alias('SalesTerritory'),
        normalize_text('Region').alias('Region'),
        normalize_text('Subregion').alias('Subregion'),
        F.col('Latest_Recorded_Population').cast(IntegerType()).alias('LatestRecordedPopulation')
    )
)

dim_stock_item = (
    stock_item_silver
    .join(referenced_stock_item_keys, on='Stock_Item_Key', how='inner')
    .withColumn('RowPriority', F.row_number().over(stock_item_window))
    .filter(F.col('RowPriority') == 1)
    .select(
        F.col('Stock_Item_Key').cast(IntegerType()).alias('StockItemKey'),
        F.col('WWI_Stock_Item_ID').cast(IntegerType()).alias('WWIStockItemId'),
        normalize_text('Stock_Item').alias('StockItemName'),
        normalize_text('Color').alias('Color'),
        normalize_text('Selling_Package').alias('SellingPackage'),
        normalize_text('Buying_Package').alias('BuyingPackage'),
        normalize_text('Brand').alias('Brand'),
        normalize_text('Size').alias('Size'),
        F.col('Lead_Time_Days').cast(IntegerType()).alias('LeadTimeDays'),
        F.col('Quantity_Per_Outer').cast(IntegerType()).alias('QuantityPerOuter'),
        F.col('Is_Chiller_Stock').cast(BooleanType()).alias('IsChillerStock'),
        F.col('Tax_Rate').cast(RATE_DECIMAL_TYPE).alias('TaxRate'),
        F.col('Unit_Price').cast(MONEY_DECIMAL_TYPE).alias('UnitPrice'),
        F.col('Recommended_Retail_Price').cast(MONEY_DECIMAL_TYPE).alias('RecommendedRetailPrice'),
        F.col('Typical_Weight_Per_Unit').cast(WEIGHT_DECIMAL_TYPE).alias('TypicalWeightPerUnit'),
        normalize_product_category('Product_Category').alias('ProductCategory')
    )
)

display(dim_city.limit(20))
display(dim_stock_item.limit(20))

## Gold DimDate の作成
`silver.Date` を基に、Power BI のタイムインテリジェンスで扱いやすい `gold.DimDate` を作成します。

設計方針:
- 一意な Date テーブルにする
- カレンダー年度と会計年度の両方で分析しやすい属性を持たせる
- 並び替えに使えるキー列を持たせる
- `FactSales[InvoiceDate]` / `FactSales[DeliveryDate]` と `DimDate[Date]` を直接関連付けやすい形にする
- Power BI で解釈がぶれないよう、派生列も型を明示的に固定する

In [ ]:
dim_date = (
    date_silver
    .select(
        F.col('Date').alias('Date'),
        F.col('Day_Number').cast(IntegerType()).alias('Day_Number'),
        F.col('Day').cast(IntegerType()).alias('Day'),
        normalize_text('Month').alias('Month'),
        normalize_text('Short_Month').alias('Short_Month'),
        F.col('Calendar_Month_Number').cast(IntegerType()).alias('Calendar_Month_Number'),
        normalize_text('Calendar_Month_Label').alias('Calendar_Month_Label'),
        F.col('Calendar_Year').cast(IntegerType()).alias('Calendar_Year'),
        normalize_text('Calendar_Year_Label').alias('Calendar_Year_Label'),
        F.col('Fiscal_Month_Number').cast(IntegerType()).alias('Fiscal_Month_Number'),
        normalize_text('Fiscal_Month_Label').alias('Fiscal_Month_Label'),
        F.col('Fiscal_Year').cast(IntegerType()).alias('Fiscal_Year'),
        normalize_text('Fiscal_Year_Label').alias('Fiscal_Year_Label'),
        F.col('ISO_Week_Number').cast(IntegerType()).alias('ISO_Week_Number')
    )
    .withColumn('DateKey', F.date_format('Date', 'yyyyMMdd').cast(IntegerType()))
    .withColumn('Calendar_Quarter_Number', F.quarter('Date').cast(IntegerType()))
    .withColumn('Calendar_Quarter_Label', F.concat(F.lit('CQ'), F.quarter('Date')))
    .withColumn('Calendar_Year_Quarter_Key', ((F.col('Calendar_Year') * F.lit(10)) + F.quarter('Date')).cast(IntegerType()))
    .withColumn('Calendar_Year_Quarter_Label', F.concat(F.lit('CY'), F.col('Calendar_Year'), F.lit('-Q'), F.quarter('Date')))
    .withColumn('Calendar_Year_Month_Key', F.date_format('Date', 'yyyyMM').cast(IntegerType()))
    .withColumn('Calendar_Year_Month_Label', F.concat(F.lit('CY'), F.col('Calendar_Year'), F.lit('-'), F.col('Short_Month')))
    .withColumn('Fiscal_Quarter_Number', F.ceil(F.col('Fiscal_Month_Number') / F.lit(3)).cast(IntegerType()))
    .withColumn('Fiscal_Quarter_Label', F.concat(F.lit('FQ'), F.ceil(F.col('Fiscal_Month_Number') / F.lit(3))))
    .withColumn('Fiscal_Year_Quarter_Key', ((F.col('Fiscal_Year') * F.lit(10)) + F.ceil(F.col('Fiscal_Month_Number') / F.lit(3))).cast(IntegerType()))
    .withColumn('Fiscal_Year_Quarter_Label', F.concat(F.lit('FY'), F.col('Fiscal_Year'), F.lit('-Q'), F.ceil(F.col('Fiscal_Month_Number') / F.lit(3))))
    .withColumn('Fiscal_Year_Month_Key', ((F.col('Fiscal_Year') * F.lit(100)) + F.col('Fiscal_Month_Number')).cast(IntegerType()))
    .withColumn('Day_Of_Week_Number_Monday_Start', F.expr("((dayofweek(Date) + 5) % 7) + 1").cast(IntegerType()))
    .withColumn('Day_Of_Week_Name', F.date_format('Date', 'E'))
    .withColumn('Is_Weekend', F.dayofweek('Date').isin([1, 7]).cast(BooleanType()))
    .withColumn('Is_Month_Start', (F.dayofmonth('Date') == 1).cast(BooleanType()))
    .withColumn('Is_Month_End', (F.last_day('Date') == F.col('Date')).cast(BooleanType()))
    .withColumn('Days_From_Today', F.datediff(F.current_date(), F.col('Date')).cast(IntegerType()))
    .withColumn('Calendar_Month_Sort_Key', ((F.col('Calendar_Year') * F.lit(100)) + F.col('Calendar_Month_Number')).cast(IntegerType()))
    .withColumn('Fiscal_Month_Sort_Key', ((F.col('Fiscal_Year') * F.lit(100)) + F.col('Fiscal_Month_Number')).cast(IntegerType()))
)

display(dim_date.orderBy('Date').limit(20))

## FactSales の作成
監査列や Silver 管理列は落とし、分析向けのキー列と指標列に整理します。

Power BI 側で `DimDate[Date]` と直接リレーションできるよう、Fact にも `InvoiceDate` / `DeliveryDate` の `date` 型列を持たせます。
金額と税率は `decimal`、件数は `int` に固定します。
`CustomerKey` / `BillToCustomerKey` / `SalespersonKey` は将来ディメンションを追加できるよう Fact に残します。

In [ ]:
fact_sales = (
    sale_silver
    .select(
        F.col('Sale_Key').cast(IntegerType()).alias('SaleKey'),
        F.col('City_Key').cast(IntegerType()).alias('CityKey'),
        F.col('Stock_Item_Key').cast(IntegerType()).alias('StockItemKey'),
        F.col('Invoice_Date_Key').cast('date').alias('InvoiceDate'),
        F.col('Delivery_Date_Key').cast('date').alias('DeliveryDate'),
        F.col('Customer_Key').cast(IntegerType()).alias('CustomerKey'),
        F.col('Bill_To_Customer_Key').cast(IntegerType()).alias('BillToCustomerKey'),
        F.col('Salesperson_Key').cast(IntegerType()).alias('SalespersonKey'),
        F.col('WWI_Invoice_ID').cast(IntegerType()).alias('WWIInvoiceId'),
        F.col('Quantity').cast(IntegerType()).alias('Quantity'),
        F.col('Unit_Price').cast(MONEY_DECIMAL_TYPE).alias('UnitPrice'),
        F.col('Tax_Rate').cast(RATE_DECIMAL_TYPE).alias('TaxRate'),
        F.col('Total_Excluding_Tax').cast(MONEY_DECIMAL_TYPE).alias('TotalExcludingTax'),
        F.col('Tax_Amount').cast(MONEY_DECIMAL_TYPE).alias('TaxAmount'),
        F.col('Profit').cast(MONEY_DECIMAL_TYPE).alias('Profit'),
        F.col('Total_Including_Tax').cast(MONEY_DECIMAL_TYPE).alias('TotalIncludingTax'),
        F.col('Total_Dry_Items').cast(IntegerType()).alias('TotalDryItems'),
        F.col('Total_Chiller_Items').cast(IntegerType()).alias('TotalChillerItems')
    )
)

display(fact_sales.limit(20))

## Gold スキーマへの保存
作成した `DimCity`、`DimStockItem`、`DimDate`、`FactSales` を Gold スキーマに保存します。

保存後は、Power BI のセマンティックモデルから参照する前提でスキーマも確認します。

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")

dim_city.write.mode('overwrite').save(DIM_CITY_PATH)
dim_stock_item.write.mode('overwrite').save(DIM_STOCK_ITEM_PATH)
dim_date.write.mode('overwrite').save(DIM_DATE_PATH)
fact_sales.write.mode('overwrite').save(FACT_SALES_PATH)

save_results = [
    (DIM_CITY_TABLE, DIM_CITY_PATH, dim_city.count()),
    (DIM_STOCK_ITEM_TABLE, DIM_STOCK_ITEM_PATH, dim_stock_item.count()),
    (DIM_DATE_TABLE, DIM_DATE_PATH, dim_date.count()),
    (FACT_SALES_TABLE, FACT_SALES_PATH, fact_sales.count())
]
display(spark.createDataFrame(save_results, ['TableName', 'Path', 'RowCount']))

## 検証
Fact と Gold ディメンションの参照整合性を確認します。`InvoiceDate` / `DeliveryDate` が `DimDate[Date]` に接続できることに加えて、Gold テーブルのスキーマも確認します。

In [ ]:
missing_city_refs = fact_sales.join(dim_city.select('CityKey'), on='CityKey', how='left_anti').count()
missing_stock_item_refs = fact_sales.join(dim_stock_item.select('StockItemKey'), on='StockItemKey', how='left_anti').count()
missing_invoice_date_refs = (
    fact_sales
    .join(dim_date.select(F.col('Date').alias('InvoiceDate')), on='InvoiceDate', how='left_anti')
    .count()
)
missing_delivery_date_refs = (
    fact_sales
    .filter(F.col('DeliveryDate').isNotNull())
    .join(dim_date.select(F.col('Date').alias('DeliveryDate')), on='DeliveryDate', how='left_anti')
    .count()
)

validation_results = [
    ('FactSales', fact_sales.count()),
    ('DimCity', dim_city.count()),
    ('DimStockItem', dim_stock_item.count()),
    ('DimDate', dim_date.count()),
    ('Missing City references', missing_city_refs),
    ('Missing StockItem references', missing_stock_item_refs),
    ('Missing InvoiceDate references', missing_invoice_date_refs),
    ('Missing DeliveryDate references', missing_delivery_date_refs)
]
display(spark.createDataFrame(validation_results, ['CheckName', 'Value']))

print('DimCity schema')
dim_city.printSchema()
print('DimStockItem schema')
dim_stock_item.printSchema()
print('DimDate schema')
dim_date.printSchema()
print('FactSales schema')
fact_sales.printSchema()

## 補足
- `City` と `Stock_Item` は Fact が参照するキーだけを Gold に持ち込みます。
- `silver.Date` を基に `gold.DimDate` を生成し、Power BI のタイムインテリジェンスに使いやすい属性を追加しています。
- `FactSales` は `InvoiceDate` / `DeliveryDate` の `date` 型列を持ち、Power BI では `DimDate[Date]` と直接リレーションする想定です。
- 金額・税率・重量は Power BI のセマンティックモデルで扱いやすいよう `decimal` に固定しています。
- `Valid_From`, `Valid_To`, `Lineage_Key`, `ProcessedTimestamp`, `DataQualityFlag` は分析公開モデルから除外しています。
- `Customer`, `Bill To Customer`, `Salesperson` の Gold ディメンションを追加したい場合は、対応する Silver ソースが別途必要です。